In [89]:
import pandas as pd

In [90]:
calendar = pd.read_csv("../data/raw/calendar.csv")
prices = pd.read_csv("../data/raw/sell_prices.csv")

In [91]:
prices["store_id"]= prices["store_id"].astype("category")
prices["item_id"]= prices["item_id"].astype("category")
prices["wm_yr_wk"]= prices["wm_yr_wk"].astype("int16")
prices["sell_price"]= prices["sell_price"].astype("float32")

In [92]:
prices.memory_usage(deep=True).sum()/1024**3

np.float64(0.05751755926758051)

In [93]:
df_chunks = pd.read_csv("../data/raw/sales_train_validation.csv",chunksize=1000)
sales = next(df_chunks)

In [94]:
sales.shape

(1000, 1919)

In [95]:
sales.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [96]:
daily_columns = [col for col in sales.columns if col.startswith("d_")]
id_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

sales_long = sales.melt(
    id_vars=id_columns, value_vars=daily_columns, var_name="d", value_name="sales"
)


In [97]:
sales_long.shape

(1913000, 8)

In [98]:
sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [99]:
sales_long = pd.merge(sales_long, calendar, on="d", how="left", validate="many_to_one")
sales_long["snap"] = sales_long["snap_CA"].where(
    sales_long["state_id"] == "CA",
    sales_long["snap_TX"].where(
        sales_long["state_id"] == "TX",
        sales_long["snap_WI"].where(sales_long["state_id"] == "WI"),
    ),
)

# sales_long = sales_long.sort_values(by=["id","item_id","store_id","date"]).reset_index(drop=True)


In [100]:
sales_long.drop(columns=["d","snap_CA","snap_TX","snap_WI"], inplace=True)

In [101]:
sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0


In [102]:
sales_long.shape

(1913000, 18)

In [109]:
sales_long = pd.merge(sales_long,prices, on =["store_id","item_id","wm_yr_wk"], how="left", validate="many_to_one")


In [111]:
sales_long.shape

(1913000, 20)

In [105]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [106]:
prices[(prices["wm_yr_wk"]==11101) & (prices["item_id"]=="HOBBIES_1_001")]

,store_id,item_id,wm_yr_wk,sell_price


In [31]:
prices[(prices["item_id"]=="HOBBIES_1_001")]

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
...,...,...,...,...
6145173,WI_3,HOBBIES_1_001,11617,8.38
6145174,WI_3,HOBBIES_1_001,11618,8.38
6145175,WI_3,HOBBIES_1_001,11619,8.38
6145176,WI_3,HOBBIES_1_001,11620,8.38


In [38]:
s = prices.groupby(["store_id","item_id"])[["wm_yr_wk"]].min().reset_index()
s[s["item_id"]=="HOBBIES_1_001"]

,store_id,item_id,wm_yr_wk
1437,CA_1,HOBBIES_1_001,11325
4486,CA_2,HOBBIES_1_001,11327
7535,CA_3,HOBBIES_1_001,11326
10584,CA_4,HOBBIES_1_001,11325
13633,TX_1,HOBBIES_1_001,11325
16682,TX_2,HOBBIES_1_001,11327
19731,TX_3,HOBBIES_1_001,11327
22780,WI_1,HOBBIES_1_001,11325
25829,WI_2,HOBBIES_1_001,11327
28878,WI_3,HOBBIES_1_001,11328


In [ ]:
sales_long[(sales_long["item_id"]=="HOBBIES_1_001")]

,id,item_id,dept_id,cat_id,store_id,state_id,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1908,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,2016-04-20,11612,Wednesday,5,4,2016,NaN,NaN,NaN,NaN,0,0,0
1909,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,3,2016-04-21,11612,Thursday,6,4,2016,NaN,NaN,NaN,NaN,0,0,0
1910,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2016-04-22,11612,Friday,7,4,2016,NaN,NaN,NaN,NaN,0,0,0
1911,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,2016-04-23,11613,Saturday,1,4,2016,NaN,NaN,NaN,NaN,0,0,0


In [23]:
prices.sort_values(by=["wm_yr_wk","store_id","item_id"]).head(30)

,store_id,item_id,wm_yr_wk,sell_price
368746,CA_1,FOODS_1_001,11101,2.00
369028,CA_1,FOODS_1_002,11101,7.88
369310,CA_1,FOODS_1_003,11101,2.88
369817,CA_1,FOODS_1_005,11101,2.94
370099,CA_1,FOODS_1_006,11101,1.97
371064,CA_1,FOODS_1_011,11101,2.28
371602,CA_1,FOODS_1_013,11101,1.25
372326,CA_1,FOODS_1_016,11101,1.96
372832,CA_1,FOODS_1_018,11101,1.00
373114,CA_1,FOODS_1_019,11101,0.97


In [13]:
prices.shape

(6841121, 4)

In [ ]:
sales_df_prices = pd.merge(
    sales_long,
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)


In [108]:
sales_df_prices.head(30)

,id,item_id,dept_id,cat_id,store_id,state_id,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap,sell_price_x,sell_price_y
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,12,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0.46,0.46
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,2,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,1.56,1.56
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,3.17,3.17


In [11]:
calendar

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1964,2016-06-15,11620,Wednesday,5,6,2016,d_1965,NaN,NaN,NaN,NaN,0,1,1
1965,2016-06-16,11620,Thursday,6,6,2016,d_1966,NaN,NaN,NaN,NaN,0,0,0
1966,2016-06-17,11620,Friday,7,6,2016,d_1967,NaN,NaN,NaN,NaN,0,0,0
1967,2016-06-18,11621,Saturday,1,6,2016,d_1968,NaN,NaN,NaN,NaN,0,0,0
